In [1]:
import json
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed
)


c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_seed(42)

LABEL_LIST = ["O", "B-COMP", "I-COMP"]

LABEL2ID = {
    "O": 0,
    "B-COMP": 1,
    "I-COMP": 2
}

ID2LABEL = {
    0: "O",
    1: "B-COMP",
    2: "I-COMP"
}

In [3]:
def load_ner_json(path):

    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    tokens = []
    ner_tags = []

    for r in records:

        tokens.append(r["tokens"])

        # file có thể là labels hoặc ner_tags
        labels = r.get("labels", r.get("ner_tags"))

        ner_tags.append(
            [LABEL2ID[label] for label in labels]
        )

    return Dataset.from_dict({
        "tokens": tokens,
        "ner_tags": ner_tags
    })

In [4]:
train_dataset = load_ner_json(
    "../data/processed/ner_train.json"
)

test_dataset = load_ner_json(
    "../data/processed/ner_test.json"
)

print(train_dataset[0])

{'tokens': ['đặt', 'hàng', 'mấy', 'ngày', 'không', 'thấy', 'chuẩn', 'bị', ',', 'nhắn', 'tin', 'hỏi', 'thì', 'không', 'trả', 'lời', 'mấy', 'ngày', 'liền', ',', 'quá', 'tệ'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [5]:
MODEL_NAME = "vinai/phobert-base-v2"

In [6]:
# Load tokenizer for PhoBERT
# Nếu model là vinai/phobert-base-v2 thì tokenizer sẽ xử lý đúng các token tiếng Việt.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

In [7]:
def tokenize_and_align_labels(examples):

    MAX_LEN = 256

    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for tokens, labels in zip(
        examples["tokens"],
        examples["ner_tags"]
    ):

        input_ids = []
        label_ids = []

        # token đặc biệt đầu câu
        input_ids.append(tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, label in zip(tokens, labels):

            # tokenize từng word
            word_tokens = tokenizer.tokenize(word)

            # convert sang ids
            word_ids = tokenizer.convert_tokens_to_ids(word_tokens)

            # nếu tokenizer trả về rỗng thì bỏ qua
            if len(word_ids) == 0:
                continue

            input_ids.extend(word_ids)

            # token đầu tiên giữ label thật
            label_ids.append(label)

            # subword phía sau -> -100
            for _ in range(len(word_ids) - 1):
                label_ids.append(-100)

        # token đặc biệt cuối câu
        input_ids.append(tokenizer.sep_token_id)
        label_ids.append(-100)

        # truncate nếu vượt MAX_LEN
        if len(input_ids) > MAX_LEN:
            input_ids = input_ids[:MAX_LEN - 1] + [tokenizer.sep_token_id]
            label_ids = label_ids[:MAX_LEN - 1] + [-100]

        attention_mask = [1] * len(input_ids)

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    }

In [8]:
train_tokenized = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_tokenized = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)

Map: 100%|██████████| 100/100 [00:00<00:00, 1749.49 examples/s]


In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24410.70it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from seqeval.metrics import precision_score, recall_score, f1_score

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        current_predictions = []
        current_labels = []
        for pred, lab in zip(prediction, label):
            if lab != -100:  # Bỏ qua các token đặc biệt và subword phụ
                current_predictions.append(ID2LABEL[pred])
                current_labels.append(ID2LABEL[lab])
        true_predictions.append(current_predictions)
        true_labels.append(current_labels)
    
    # Tính toán trực tiếp bằng thư viện seqeval
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [12]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [13]:
training_args = TrainingArguments(
    output_dir="../models/phobert_ner_baseline",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    dataloader_pin_memory=False,

    logging_steps=50,

    report_to="none",

    seed=42
)

In [14]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.154776,0.014313,0.000000,0.000000,0.000000
2,0.014473,0.010907,0.000000,0.000000,0.000000
3,0.012316,0.009841,0.000000,0.000000,0.000000
4,0.011280,0.009232,0.000000,0.000000,0.000000
5,0.010706,0.009019,0.000000,0.000000,0.000000


c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:159: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_divisio

TrainOutput(global_step=250, training_loss=0.04071015954017639, metrics={'train_runtime': 780.3581, 'train_samples_per_second': 2.563, 'train_steps_per_second': 0.32, 'total_flos': 97191023910480.0, 'train_loss': 0.04071015954017639, 'epoch': 5.0})

In [16]:
results = trainer.evaluate()

print("=" * 50)
print("PhoBERT NER Baseline")
print("=" * 50)

print(f"Precision : {results['eval_precision']:.4f}")
print(f"Recall    : {results['eval_recall']:.4f}")
print(f"F1-score  : {results['eval_f1']:.4f}")

c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python313\Lib\site-packages\seqeval\metrics\v1.py:159: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.010706,0.014313,5,0.000000,0.000000,0.000000


PhoBERT NER Baseline
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000


In [17]:
trainer.save_model("../models/phobert_ner_baseline")

tokenizer.save_pretrained(
    "../models/phobert_ner_baseline"
)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


('../models/phobert_ner_baseline\\tokenizer_config.json',
 '../models/phobert_ner_baseline\\vocab.txt',
 '../models/phobert_ner_baseline\\bpe.codes',
 '../models/phobert_ner_baseline\\added_tokens.json')

In [18]:
baseline_results = {
    "model": MODEL_NAME,
    "task": "NER",
    "labels": LABEL_LIST,
    "precision": round(results["eval_precision"], 4),
    "recall": round(results["eval_recall"], 4),
    "f1": round(results["eval_f1"], 4)
}

with open(
    "../models/phobert_ner_baseline/baseline_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_results,
        f,
        ensure_ascii=False,
        indent=2
    )